<a href="https://colab.research.google.com/github/stfnnnnnnn/karl-mangahas-flyrank/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/stfnnnnnnn/1st-act/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
pip -q install duckdb huggingface_hub

In [2]:
import os, getpass

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass()

In [6]:
import duckdb

con = duckdb.connect()

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf
    (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
    """
)


In [7]:
REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients":
        f"read_parquet('{REL}/dim_clients.parquet')",

    "dim_content":
        f"read_parquet('{REL}/dim_content.parquet')",

    "fact_daily":
        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",

    "fact_daily_sample":
        f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",

    "fact_query_90d":
        f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

In [5]:
for name, src in TABLES.items():
    n = con.sql(f"SELECT COUNT(*) FROM {src}").fetchone()[0]
    print(f"{name:22} {n:,} rows")

dim_clients            104 rows
dim_content            519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily             78,835,655 rows
fact_daily_sample      11,694,072 rows
fact_query_90d         2,414,248 rows


In [8]:
import pandas as pd

# Same basic 30-day historical / 30-day later-outcome design established in W03.
features = con.sql(f"""
WITH bounds AS (
    SELECT MAX(report_date) AS end_d
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
),
windowed AS (
    SELECT
        f.client_hash_id,
        f.content_hash_id,

        SUM(CASE
            WHEN f.report_date > b.end_d - INTERVAL 30 DAY
            THEN f.gsc_impressions ELSE 0 END) AS imp_last30,

        SUM(CASE
            WHEN f.report_date <= b.end_d - INTERVAL 30 DAY
            THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,

        SUM(CASE
            WHEN f.report_date > b.end_d - INTERVAL 30 DAY
            THEN f.gsc_clicks ELSE 0 END) AS clk_last30,

        SUM(CASE
            WHEN f.report_date <= b.end_d - INTERVAL 30 DAY
            THEN f.gsc_clicks ELSE 0 END) AS clk_prev30,

        AVG(CASE
            WHEN f.report_date <= b.end_d - INTERVAL 30 DAY
            THEN f.gsc_avg_position END) AS pos_prev30,

        AVG(CASE
            WHEN f.report_date > b.end_d - INTERVAL 30 DAY
            THEN f.gsc_avg_position END) AS pos_last30

    FROM {TABLES['fact_daily']} f, bounds b
    WHERE f.report_date > b.end_d - INTERVAL 60 DAY
      AND f.report_date <= b.end_d
    GROUP BY f.client_hash_id, f.content_hash_id
    HAVING imp_prev30 >= 100
)
SELECT *
FROM windowed
""").df()

features["is_declining"] = (
    features["imp_last30"] < 0.8 * features["imp_prev30"]
).astype(int)

print(f"Rows: {len(features):,}")
print(f"Clients: {features['client_hash_id'].nunique():,}")
print(f"Decline base rate: {features['is_declining'].mean():.1%}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 82,025
Clients: 37
Decline base rate: 26.9%


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**The rule, in plain words.**

>A page deserves earlier review if it demonstrated meaningful search demand during the previous 30-day feature window but was already ranking outside page one during that same historical period. The reasoning is simple: pages with established search visibility represent more meaningful review opportunities than pages with little historical demand, while weaker historical position provides an additional signal that the page may deserve attention.

>This baseline is not intended to predict decline perfectly or determine what action should be taken. It provides a simple and transparent starting rule that I can inspect manually and later compare against a learned model. The rule uses only information available in the historical feature window.

**Signal 1 – Search position (position-based review logic)**

>The first signal examines whether pages in weaker **historical search-position tiers** were more likely to experience decline during the later outcome window. Pages are grouped using their previous-window average position into the standard tiers (`top_3`, `page_1`, `striking`, `page_3_5`, and `deep`), then compared using the later observed decline rate within each group.

>This helps determine whether historical search position provides enough directional information to justify including it in the baseline rule.

**Verdict: MIXED**

>The observed decline rates varied across position tiers rather than increasing consistently as historical rankings became weaker. The highest decline rate occurred in the `striking` tier (26.2%, n = 17,366), while `page_3_5` (19.6%, n = 15,635) and `deep` (19.7%, n = 1,630) showed lower rates.

>This means historical position contains some useful separation, but the relationship is **not monotonic**. A weaker position by itself is therefore not enough to identify pages that will later decline. I keep position in the baseline as a simple prioritization signal rather than treating it as a standalone predictor of decline.

**Signal 2 – Average CTR across historical search-position tiers**

>The second signal examines how average **historical CTR** differs across the same historical position tiers. Because click-through rate naturally depends on search position, comparing CTR across position groups provides useful context for whether position reflects meaningful differences in how much search visibility turns into clicks.

**Verdict: CONFIRMED**

>Average CTR decreased steadily from 0.352% (n = 6,527) in the `top_3` tier to 0.083% (n = 1,630) in the `deep` tier, with each successive position tier showing lower observed CTR.

>This confirms the expected relationship between historical search position and click capture in this development partition. However, it does **not** show that weaker position causes future decline. Together with the mixed decline-rate result above, it supports using position as a transparent review signal while keeping the baseline intentionally simple.

**Reason code:** `visible_weak_position`

>I use one reason code because this is a single-rule baseline rather than a multi-signal diagnostic system. The code means that the page had meaningful historical visibility and a historical average position outside page one.

**Action label:** `flag_for_review` when the baseline score is above 0; otherwise `no_action`.

`flag_for_review` means the page should receive earlier human inspection. It does not mean that the page must be refreshed, rewritten, or otherwise changed.

In [9]:
import numpy as np

# Historical search-position tiers
pos_bins = [-1, 0, 3, 10, 20, 50, np.inf]
pos_labels = [
    "no_data",
    "top_3",
    "page_1",
    "striking",
    "page_3_5",
    "deep",
]

features["position_tier"] = pd.cut(
    features["pos_prev30"].fillna(0),
    bins=pos_bins,
    labels=pos_labels,
)

# Historical CTR from the same previous 30-day feature window
features["ctr_prev30"] = (
    features["clk_prev30"]
    / features["imp_prev30"].replace(0, np.nan)
)

# Exclude pages without historical position data from the tier comparison
have_position = features[
    features["position_tier"] != "no_data"
].copy()

# Signal 1:
# Did later decline rates differ across HISTORICAL position tiers?
bucket_position = (
    have_position
    .groupby("position_tier", observed=True)["is_declining"]
    .agg(
        n="count",
        decline_rate="mean",
    )
    .reindex([
        "top_3",
        "page_1",
        "striking",
        "page_3_5",
        "deep",
    ])
)

print("Signal 1 - historical position vs later decline rate")
print(bucket_position)

# Signal 2:
# How did historical CTR differ across HISTORICAL position tiers?
bucket_ctr = (
    have_position
    .groupby("position_tier", observed=True)["ctr_prev30"]
    .agg(
        n="count",
        avg_ctr="mean",
    )
    .reindex([
        "top_3",
        "page_1",
        "striking",
        "page_3_5",
        "deep",
    ])
)

print("\nSignal 2 - historical position vs historical average CTR")
print(bucket_ctr)

Signal 1 - position vs decline rate
                   n  decline_rate
position_tier                     
top_3           6527      0.216179
page_1         37094      0.240524
striking       17366      0.261891
page_3_5       15635      0.196226
deep            1630      0.196933

Signal 2 - position vs average CTR
                   n   avg_ctr
position_tier                 
top_3           6527  0.003518
page_1         37094  0.003226
striking       17366  0.002606
page_3_5       15635  0.001800
deep            1630  0.000827


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

>The baseline stays deliberately simple. A page receives a positive review score only when it satisfies the two historical conditions I wrote above: at least 300 prior impressions and prior average position worse than 10.

>Among eligible pages, I rank by `imp_prev30`, so pages with more demonstrated historical visibility are reviewed first.

>The later outcome fields remain in the dataframe only so I can evaluate whether the rule happened to prioritize pages that subsequently met the decline definition. They do not contribute to the score.


In [10]:
import os

features["is_visible"] = (
    features["imp_prev30"] >= 300
).astype(int)

features["is_weak_position"] = (
    features["pos_prev30"].notna()
    & (features["pos_prev30"] > 10)
).astype(int)

features["baseline_score"] = (
    features["is_visible"]
    * features["is_weak_position"]
    * features["imp_prev30"]
)

features["reason_code"] = np.where(
    features["baseline_score"] > 0,
    "visible_weak_position",
    "not_flagged"
)

features["action"] = np.where(
    features["baseline_score"] > 0,
    "flag_for_review",
    "no_action"
)

ranked = (
    features
    .sort_values("baseline_score", ascending=False)
    .reset_index(drop=True)
)

os.makedirs("work/outputs", exist_ok=True)

out_cols = [
    "client_hash_id",
    "content_hash_id",
    "baseline_score",
    "reason_code",
    "action",
    "imp_prev30",
    "clk_prev30",
    "pos_prev30",
    "is_declining",
]

ranked[out_cols].to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

n_flagged = int(
    (ranked["baseline_score"] > 0).sum()
)

base_rate = ranked["is_declining"].mean()
p20 = ranked["is_declining"].head(20).mean()
p50 = ranked["is_declining"].head(50).mean()

print(
    f"Wrote {len(ranked):,} rows. "
    f"{n_flagged:,} flagged for review "
    f"({n_flagged / len(ranked):.1%})."
)

print(
    f"base rate: {base_rate:.1%} | "
    f"precision@20: {p20:.1%} | "
    f"precision@50: {p50:.1%}"
)

ranked[out_cols].head(10)

Wrote 82,025 rows. 18,442 flagged for review (22.5%).
base rate: 26.9% | precision@20: 5.0% | precision@50: 10.0%


,client_hash_id,content_hash_id,baseline_score,reason_code,action,imp_prev30,clk_prev30,pos_prev30,is_declining
0,client_23a62021009f63c4,content_e8a52cf3d5988c07,170127.0,visible_weak_position,flag_for_review,170127.0,654.0,13.843213,0
1,client_23a62021009f63c4,content_36e53e9c707674fc,108522.0,visible_weak_position,flag_for_review,108522.0,251.0,34.025098,0
2,client_23a62021009f63c4,content_df47d1b976106de4,90417.0,visible_weak_position,flag_for_review,90417.0,108.0,18.133751,0
3,client_fef1a8f436438636,content_84a6bf3578312e90,83691.0,visible_weak_position,flag_for_review,83691.0,78.0,19.985366,0
4,client_23a62021009f63c4,content_5e1c049f62e33b11,76602.0,visible_weak_position,flag_for_review,76602.0,122.0,16.619271,0
5,client_fef1a8f436438636,content_ba462518dad435fc,72994.0,visible_weak_position,flag_for_review,72994.0,41.0,27.270923,0
6,client_23a62021009f63c4,content_3df3f32f3fd58dea,71553.0,visible_weak_position,flag_for_review,71553.0,160.0,24.862457,0
7,client_23a62021009f63c4,content_b51957d7f4abe47e,60784.0,visible_weak_position,flag_for_review,60784.0,36.0,27.590981,0
8,client_23a62021009f63c4,content_bdf60c86117079be,54633.0,visible_weak_position,flag_for_review,54633.0,8.0,33.456936,0
9,client_fef1a8f436438636,content_0aaa197051f58d6f,53243.0,visible_weak_position,flag_for_review,53243.0,36.0,34.602393,0


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*


The baseline produces a ranked queue rather than a binary prediction, so the highest-ranked pages deserve manual inspection before drawing conclusions. For each of the top ten pages, I review why the rule selected it and identify one realistic situation where the recommendation could be misleading. This helps evaluate whether the rule is behaving as intended before comparing it against a machine learning model.

### Top-20 review

The table above provides the required review information for each of the top 20 pages: the action, reason code, confidence note, and a condition that could make the recommendation wrong.

All 20 pages were flagged by the same transparent baseline rule: they had meaningful historical visibility and an average historical search position outside page one. Because the baseline score is equal to `imp_prev30` for pages that satisfy both conditions, the highest-ranked pages are the qualifying pages with the greatest previous-window impressions.

The review also shows an important limitation of the rule. **Only 1 of the top 20 pages later met the decline definition**, while the other 19 did not. This means that high historical visibility combined with weak historical position was not enough to reliably identify pages that subsequently declined.

For example, `content_e8a52cf3d5988c07` ranked first because it had 170,127 previous impressions and an average historical position of 13.84, but its later outcome was not classified as declining. The same pattern appears across most of the top-ranked pages: they represent high-visibility pages that may be reasonable to inspect, but the baseline frequently flags pages that do not later decline.

The main weakness is that the rule cannot distinguish a genuine early decline risk from other reasons a high-visibility page may rank outside page one, such as competition, seasonality, search intent, consolidation, or normal variation. This is exactly the limitation that a learned model would need to improve on in the next stage.

In [11]:
top20 = ranked.head(20).copy()

top20["confidence_note"] = np.where(
    top20["imp_prev30"] >= 1000,
    "higher historical visibility",
    "moderate historical visibility"
)

top20["what_would_make_it_wrong"] = (
    "competition, seasonality, intent mismatch, consolidation, "
    "or another context the simple rule cannot observe"
)

top20_cols = [
    "client_hash_id",
    "content_hash_id",
    "baseline_score",
    "reason_code",
    "action",
    "imp_prev30",
    "clk_prev30",
    "pos_prev30",
    "is_declining",
    "confidence_note",
    "what_would_make_it_wrong",
]

top20[top20_cols]


,client_hash_id,content_hash_id,baseline_score,reason_code,action,imp_prev30,clk_prev30,pos_prev30,is_declining,confidence_note,what_would_make_it_wrong
0,client_23a62021009f63c4,content_e8a52cf3d5988c07,170127.0,visible_weak_position,flag_for_review,170127.0,654.0,13.843213,0,higher historical visibility,"competition, seasonality, intent mismatch, con..."
1,client_23a62021009f63c4,content_36e53e9c707674fc,108522.0,visible_weak_position,flag_for_review,108522.0,251.0,34.025098,0,higher historical visibility,"competition, seasonality, intent mismatch, con..."
2,client_23a62021009f63c4,content_df47d1b976106de4,90417.0,visible_weak_position,flag_for_review,90417.0,108.0,18.133751,0,higher historical visibility,"competition, seasonality, intent mismatch, con..."
3,client_fef1a8f436438636,content_84a6bf3578312e90,83691.0,visible_weak_position,flag_for_review,83691.0,78.0,19.985366,0,higher historical visibility,"competition, seasonality, intent mismatch, con..."
4,client_23a62021009f63c4,content_5e1c049f62e33b11,76602.0,visible_weak_position,flag_for_review,76602.0,122.0,16.619271,0,higher historical visibility,"competition, seasonality, intent mismatch, con..."
5,client_fef1a8f436438636,content_ba462518dad435fc,72994.0,visible_weak_position,flag_for_review,72994.0,41.0,27.270923,0,higher historical visibility,"competition, seasonality, intent mismatch, con..."
6,client_23a62021009f63c4,content_3df3f32f3fd58dea,71553.0,visible_weak_position,flag_for_review,71553.0,160.0,24.862457,0,higher historical visibility,"competition, seasonality, intent mismatch, con..."
7,client_23a62021009f63c4,content_b51957d7f4abe47e,60784.0,visible_weak_position,flag_for_review,60784.0,36.0,27.590981,0,higher historical visibility,"competition, seasonality, intent mismatch, con..."
8,client_23a62021009f63c4,content_bdf60c86117079be,54633.0,visible_weak_position,flag_for_review,54633.0,8.0,33.456936,0,higher historical visibility,"competition, seasonality, intent mismatch, con..."
9,client_fef1a8f436438636,content_0aaa197051f58d6f,53243.0,visible_weak_position,flag_for_review,53243.0,36.0,34.602393,0,higher historical visibility,"competition, seasonality, intent mismatch, con..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Leakage check**

>The baseline score uses only `imp_prev30` and `pos_prev30`, both from the historical feature window defined in Week 3. `imp_last30`, `clk_last30`, `pos_last30`, and `is_declining` are not rule inputs.

**Weak picks**

The baseline is intentionally limited, so I expect some weak recommendations:

1. A page can have many impressions but still be a poor opportunity if its query is extremely competitive.
2. Historical position and impressions do not explain seasonality, intent changes, or consolidation with related pages.
3. Ranking eligible pages only by prior impressions may over-prioritize very large pages even when another page has a more actionable problem.
4. The rule does not yet combine several signals or learn interactions; that is exactly what the later model comparison will test.

>These weaknesses are useful because they give me specific cases where a learned model would need to improve on the rule rather than merely produce a different score.


In [12]:
# Leakage check: the baseline must use historical fields only.
outcome_period_cols = {
    "imp_last30",
    "clk_last30",
    "pos_last30",
    "is_declining",
}

rule_input_cols = {
    "imp_prev30",
    "pos_prev30",
}

assert outcome_period_cols.isdisjoint(rule_input_cols), (
    "Leakage: an outcome-period field is feeding the baseline."
)

print("Clean: baseline inputs come only from the historical feature window.")

weak_review_cols = [
    "client_hash_id",
    "content_hash_id",
    "imp_prev30",
    "clk_prev30",
    "pos_prev30",
    "position_tier",
    "baseline_score",
    "is_declining",
]

ranked.head(20)[weak_review_cols]


Clean: baseline inputs come only from the historical feature window.


,client_hash_id,content_hash_id,imp_prev30,clk_prev30,pos_prev30,position_tier,baseline_score,is_declining
0,client_23a62021009f63c4,content_e8a52cf3d5988c07,170127.0,654.0,13.843213,striking,170127.0,0
1,client_23a62021009f63c4,content_36e53e9c707674fc,108522.0,251.0,34.025098,page_3_5,108522.0,0
2,client_23a62021009f63c4,content_df47d1b976106de4,90417.0,108.0,18.133751,page_3_5,90417.0,0
3,client_fef1a8f436438636,content_84a6bf3578312e90,83691.0,78.0,19.985366,page_3_5,83691.0,0
4,client_23a62021009f63c4,content_5e1c049f62e33b11,76602.0,122.0,16.619271,striking,76602.0,0
5,client_fef1a8f436438636,content_ba462518dad435fc,72994.0,41.0,27.270923,page_3_5,72994.0,0
6,client_23a62021009f63c4,content_3df3f32f3fd58dea,71553.0,160.0,24.862457,page_3_5,71553.0,0
7,client_23a62021009f63c4,content_b51957d7f4abe47e,60784.0,36.0,27.590981,page_3_5,60784.0,0
8,client_23a62021009f63c4,content_bdf60c86117079be,54633.0,8.0,33.456936,page_3_5,54633.0,0
9,client_fef1a8f436438636,content_0aaa197051f58d6f,53243.0,36.0,34.602393,page_3_5,53243.0,0


## Self-check

Before you submit, confirm each line honestly:

- [ - ] Every section above is filled — markdown thinking AND the code that backs it
- [ - ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ - ] No client names, URLs, or private queries anywhere
- [ - ] My claims use careful words: observed, measured, directional, decision-support
- [ - ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.